# Unsupervised Learning Project
## Behavioural Segmentation of Travel Agency / Tour Operator Bookings in the Hotel/Resort Industry

**Team (G4-P2):** José Nunes (73137) · João Lourenço (72904) · Leonor Afonso (73491)

**Course:** Unsupervised Learning · 2025/2026 · NOVA FCT, Departamento de Informática

## 1. Environment, Imports and Reproducibility

A single arbitrary master seed (`SEED = 12345`) is used for all randomised steps. Five
auxiliary seeds (`SEEDS_STABILITY`) are predeclared here so that Task 3 (stability analysis)
plugs into the same protocol without further setup.

The Silhouette score is *O(n²)*; on the full sub-population it would be prohibitively slow.
We therefore evaluate it on a fixed, seeded sub-sample (`SUBSAMPLE_N = 10_000`). The
Calinski–Harabasz and Davies–Bouldin indices are *O(n)* and are computed on the entire
sub-population.

In [ ]:
import os
import time
import hashlib
import warnings
import json
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from dataclasses import dataclass
from numpy.typing import NDArray

from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score,
)

# Reproducibility
SEED              = 12345
SEEDS_STABILITY   = [12345, 23456, 34567, 45678, 56789] 
SUBSAMPLE_N       = 10_000   
np.random.seed(SEED)

# Project layout 
# Locate PROJECT_ROOT robustly: walk up from the current working directory
# until a marker file (requirements.txt or .git) is found.

def _find_project_root(markers=('requirements.txt', '.git', 'environment.yml')):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if any((candidate / m).exists() for m in markers):
            return candidate
    return here  # fallback: cwd

PROJECT_ROOT = _find_project_root()
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'
RESULTS_DIR  = PROJECT_ROOT / 'results'
FIG_DIR      = RESULTS_DIR / 'figures'
TBL_DIR      = RESULTS_DIR / 'tables'
RPT_DIR      = RESULTS_DIR / 'reports'

for d in (FIG_DIR, TBL_DIR, RPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# plot default
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})
sns.set_style('whitegrid')

print("Environment ready.")
print(f"Project root         : {PROJECT_ROOT}")
print(f"Data directory       : {DATA_DIR}")
print(f"Results directory    : {RESULTS_DIR}")
print(f"Master seed          : {SEED}")
print(f"Stability seeds      : {SEEDS_STABILITY}")
print(f"Silhouette subsample : {SUBSAMPLE_N:,}")


## 2. Dataset Documentation and Load

| Field | Value |
|---|---|
| **Source** | Hotel Booking Demand (course release) |
| **Reference** | António, N., de Almeida, A., & Nunes, L. (2019). *Hotel booking demand datasets*. *Data in Brief*, 22, 41–49. https://doi.org/10.1016/j.dib.2018.11.126 |
| **License** | CC BY 4.0 |
| **Coverage** | Bookings due to arrive 1 Jul 2015 – 31 Aug 2017 |
| **Hotels** | H1 - Resort (Algarve, PT); H2 - City (Lisbon, PT) |
| **Unit of analysis** | one booking record |
| **Rows × cols** | 119,390 × 32 |
| **Index time** | day prior to arrival (per the reference paper) |

**Data-quality issues identified by the EDA below** *(documented before any cleaning is applied)*:

- `country` has a small share of missing values; the dataset's reference paper notes that an empty
  country usually corresponds to the resort's domestic market, so we impute `"PRT"`.
- `children` has four `NaN` rows; we impute `0`.
- `agent` and `company` are sparsely populated identifier columns; they are excluded from
  clustering as id-like high-cardinality fields.
- A small number of records have `adr ≤ 0` (complimentary or test rows) and a single record has
  `adr ≈ 5400` (data-entry error). These rows are filtered before any analysis that uses `adr`
  (post-hoc profiling and the `withADR` sensitivity variant).
- A handful of bookings have `adults + children + babies == 0`. We drop them: a booking with no
  guests is not a valid behavioural unit.
- The MD5 of the CSV is recorded below for traceability (Milestone 2 §2).


In [ ]:
DATASET_PATH = DATA_DIR / 'hotel_bookings.csv'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        f"Place hotel_bookings.csv in {DATA_DIR} (the data/raw folder of the repo)."
    )

with open(DATASET_PATH, 'rb') as f:
    DATASET_MD5 = hashlib.md5(f.read()).hexdigest()

df_raw = pd.read_csv(DATASET_PATH)
print(f"Dataset path     : {DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"Dataset MD5      : {DATASET_MD5}")
print(f"Shape            : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Memory footprint : {df_raw.memory_usage(deep=True).sum()/1e6:.1f} MB")


## RQ1, Segmentation Time and Sub-Population

### RQ1 - formal statement

> **RQ1.** *Among bookings made through Travel Agency / Tour Operator (TA/TO) intermediaries,
> are there latent behavioural groups characterised jointly by booking timing, stay design,
> party composition and commercial-channel metadata that are observable at booking time?*

### Unit of analysis and segmentation time

| Item | Choice | Rationale |
|---|---|---|
| Unit of analysis | one booking record | one row = one booking (per the dataset documentation) |
| **Segmentation time (index time)** | **immediately after the booking is recorded** | This is the moment at which the hotel would assign a booking to a segment. Any feature whose value is updated *after* this moment is excluded from clustering inputs. |
| Practical effect | `booking_changes`, `assigned_room_type`, `is_canceled`, `reservation_status`, `reservation_status_date`, `days_in_waiting_list` are NOT clustering inputs | They are post-decision or post-arrival variables under the booking-time index. |

This explicit choice is what the lecturer asked for in Correction #2 of Deliverable 1.
Under it, `booking_changes` is **profiling-only**, not a clustering input.

### Target sub-population

The inclusion rule is: include  if and only if  distribution_channel == "TA/TO"

We do not pool TA/TO with GDS, Corporate or Direct. The justifications are:

1. **Consistency with title and RQ1.** The project title explicitly names "Travel Agency and
   Tour Operator". Mixing in GDS or corporate channels would dilute the population definition.
2. **Statistical scale.** TA/TO has 97,870 records (82% of the dataset); GDS has only 193 records,
   too few to form a stable cluster on its own and small enough to dominate the loss only via
   noise if pooled.
3. **Channel homogeneity.** Within TA/TO, `market_segment` still distinguishes online vs offline
   intermediaries, groups, etc., so behavioural heterogeneity is preserved without needing GDS or
   Corporate channels in the same partition.

### Conservative interpretation

Because raw `agent` and `company` identifiers are excluded as id-like high-cardinality fields,
the discovered clusters describe **booking profiles associated with intermediary-mediated
demand** - they are *not* definitive agency or tour-operator archetypes. This caveat is recorded
here and is repeated when results are interpreted in Task 4 of the report.


In [ ]:
# Distribution channel breakdown (raw data)
fig, ax = plt.subplots(figsize=(8, 4))

ch_counts = df_raw['distribution_channel'].value_counts()
colors = ['#1565C0' if c == 'TA/TO' else '#B0BEC5' for c in ch_counts.index]
bars = ax.bar(ch_counts.index, ch_counts.values, color=colors,
              edgecolor='white', linewidth=0.8)

# annotate counts
for bar, v in zip(bars, ch_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v, f'{v:,}',
            ha='center', va='bottom', fontsize=9)

ax.set_title('Figure 1 — Distribution channels (raw data); TA/TO is the target sub-population',
             fontweight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.savefig(FIG_DIR / 'fig1_channel_distribution.png')
plt.show()


In [ ]:
# Apply the explicit inclusion rule
df = df_raw[df_raw['distribution_channel'] == 'TA/TO'].copy()

n_total = len(df_raw)
n_kept  = len(df)
print(f"Total raw records          : {n_total:,}")
print(f"TA/TO records (kept)       : {n_kept:,} ({n_kept/n_total*100:.1f}%)")
print(f"Excluded (other channels)  : {n_total - n_kept:,}")
print()
print("Excluded channels:")
print(df_raw[df_raw['distribution_channel'] != 'TA/TO']['distribution_channel']
      .value_counts().to_string())


## Exploratory Data Analysis (EDA)

### Missingness audit

Missingness is reported by attribute type. We distinguish *structural absence* (e.g. no agent
recorded for a direct booking) from genuinely missing data. Imputation choices match the
attribute type and the Milestone 2 guidance.


In [ ]:
miss = df.isnull().sum()
miss_pct = df.isnull().mean().mul(100)
miss_df = pd.DataFrame({'Missing Count': miss, 'Missing %': miss_pct.round(3)})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("Missingness Report (TA/TO sub-population)")
print("-" * 50)
print(miss_df.to_string())
miss_df.to_csv(TBL_DIR / 'missingness.csv')


In [ ]:
# Missingness heatmap on a 3,000-row sub-sample for visual clarity
miss_cols = miss_df.index.tolist()

if miss_cols:
    fig, ax = plt.subplots(figsize=(10, 3))
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(df), size=min(3_000, len(df)), replace=False)
    sns.heatmap(
        df[miss_cols].iloc[sample_idx].isnull().astype(int).T,
        cmap=['#E8F4FD', '#C62828'],
        cbar_kws={'label': 'Missing (red)', 'shrink': 0.5},
        ax=ax, yticklabels=True, xticklabels=False,
    )
    ax.set_title('Figure 2 — Missingness pattern (sample of 3,000 rows)', fontweight='bold')
    ax.set_xlabel('Records (sample)')
    plt.savefig(FIG_DIR / 'fig2_missingness.png')
    plt.show()
else:
    print("No missing values to plot.")


### Outlier and distribution audit

We summarise the numerical variables that are candidate clustering inputs (or post-hoc
profilers). Skewness and high percentiles motivate the `log1p` transform on `lead_time` and
the `withADR` sensitivity comparison.


In [ ]:
num_cols_inspect = [
    'lead_time', 'stays_in_week_nights', 'stays_in_weekend_nights',
    'adults', 'children', 'babies',
    'previous_cancellations', 'previous_bookings_not_canceled',
    'required_car_parking_spaces', 'total_of_special_requests',
    'booking_changes',  # included in EDA but excluded from clustering (Correction #2)
    'adr',              # included in EDA; only post-hoc / withADR (Correction #3)
]

stats = df[num_cols_inspect].describe(percentiles=[.25, .5, .75, .90, .95, .99]).T
stats['skew'] = df[num_cols_inspect].skew()

print("Numerical variables — descriptive statistics (TA/TO sub-population)")
print("-" * 70)
print(stats.round(2).to_string())

# Sentinel summaries for cleaning decisions later in §5
print()
print(f"adr ≤ 0           : {(df['adr'] <= 0).sum():,} rows  → drop (non-commercial)")
print(f"adr > 1000        : {(df['adr'] > 1000).sum():,} rows  → drop (extreme outlier)")
print(f"lead_time > 365   : {(df['lead_time'] > 365).sum():,} rows ({(df['lead_time'] > 365).mean()*100:.2f}%)")
print(f"adults+children+babies == 0 : {((df['adults']+df['children'].fillna(0)+df['babies']) == 0).sum():,} rows  → drop (no guests)")


In [ ]:
# Univariate distributions of key numerical features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

plot_configs = [
    ('lead_time',                'Lead Time (days)',      '#1565C0', 80),
    ('stays_in_week_nights',     'Week Night Stays',      '#2E7D32', 15),
    ('stays_in_weekend_nights',  'Weekend Night Stays',   '#E65100', 10),
    ('total_of_special_requests','Special Requests',      '#00695C',  6),
    ('booking_changes',          'Booking Changes (post-decision; PROFILING-ONLY)', '#9E9E9E', 20),
    ('adr',                      'ADR — € (PROFILING-ONLY)', '#9E9E9E', 60),
]
for i, (col, label, color, bins) in enumerate(plot_configs):
    data = df[col].dropna()
    p99 = data.quantile(0.99)
    axes[i].hist(data.clip(upper=p99), bins=bins, color=color, alpha=0.75,
                 edgecolor='white', linewidth=0.5)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_ylabel('Count')
    axes[i].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    axes[i].text(0.97, 0.95, f'skew={data.skew():.2f}', transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9, color='grey')

fig.suptitle('Figure 3 — Univariate distributions of key features '
             '(grey panels: profiling-only, NOT clustering inputs)',
             fontsize=12, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_univariate.png')
plt.show()


### Correlation and bivariate behaviour

The Pearson correlation matrix is computed on the candidate numerical block. Two notes:

- `lead_time` and `total_nights` are weakly correlated, justifying treating them as
  complementary behavioural axes.
- `previous_cancellations` and `previous_bookings_not_canceled` are weakly correlated and
  encode different aspects of the customer's past relationship with the hotel; we keep both.


In [ ]:
# Compute total_nights now (used as an engineered numeric variable, see §5)
df['total_nights'] = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df['party_size']   = df['adults'] + df['children'].fillna(0) + df['babies']

num_features_corr = [
    'lead_time', 'total_nights', 'party_size',
    'previous_cancellations', 'previous_bookings_not_canceled',
    'required_car_parking_spaces', 'total_of_special_requests',
    # post-hoc / sensitivity
    'booking_changes', 'adr',
]
corr_matrix = df[num_features_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, annot=True, fmt='.2f',
            annot_kws={'size': 9}, ax=ax)
ax.set_title('Figure 4 — Pearson correlation matrix '
             '(profiling-only variables shown for reference)', fontweight='bold')
plt.savefig(FIG_DIR / 'fig4_correlation.png')
plt.show()


In [ ]:
# Within-TA/TO market_segment behaviour
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lead time by market segment
top_ms = df['market_segment'].value_counts().index.tolist()
data_lt = [df.loc[df['market_segment'] == ms, 'lead_time'].dropna() for ms in top_ms]
bp = axes[0].boxplot(data_lt, labels=top_ms, patch_artist=True, notch=True,
                     boxprops=dict(facecolor='#90CAF9'),
                     medianprops=dict(color='#1565C0', linewidth=2))
axes[0].set_title('Lead time by market_segment (within TA/TO)')
axes[0].set_ylabel('Lead time (days)')
axes[0].set_ylim(0, 400)
axes[0].tick_params(axis='x', rotation=20)

# Total nights by market segment
data_tn = [df.loc[df['market_segment'] == ms, 'total_nights'].dropna() for ms in top_ms]
bp2 = axes[1].boxplot(data_tn, labels=top_ms, patch_artist=True, notch=True,
                      boxprops=dict(facecolor='#FFCC80'),
                      medianprops=dict(color='#E65100', linewidth=2))
axes[1].set_title('Total nights by market_segment (within TA/TO)')
axes[1].set_ylabel('Total nights')
axes[1].set_ylim(0, 15)
axes[1].tick_params(axis='x', rotation=20)

fig.suptitle('Figure 5 — RQ1 feature behaviour across market segments (TA/TO sub-population)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_bivariate_marketseg.png')
plt.show()


## 5. Feature Selection

### 5.1 Feature-selection table

This table records every variable's role under the **booking-time segmentation**.
Variables are partitioned into three roles:

| Role | Meaning |
|---|---|
| **input** | Available at booking time and meaningful for similarity → clustering input. |
| **profile** | Not used to form clusters; used only after clusters are formed. |
| **exclude** | Leakage, post-decision, raw identifier, or excluded for representation reasons. |


In [ ]:
feature_decisions = pd.DataFrame([
    # Identifiers / outcomes / post-events
    ('hotel',                          'profile',  'binary outcome-related metadata; profile only'),
    ('is_canceled',                    'exclude',  'outcome variable; not available at index time'),
    ('reservation_status',             'exclude',  'post-event status'),
    ('reservation_status_date',        'exclude',  'post-event date'),
    ('agent',                          'exclude',  'id-like high-cardinality (~3,000+ values within TA/TO)'),
    ('company',                        'exclude',  'id-like high-cardinality and very sparse'),
    ('assigned_room_type',             'exclude',  'assigned by the hotel after booking; post-decision'),
    ('booking_changes',                'profile',  'modified after the booking is recorded → not a clustering input under booking-time index (Correction #2)'),
    ('days_in_waiting_list',           'exclude',  'updated by hotel operations after booking'),

    # Numerical clustering inputs
    ('lead_time',                      'input',    'days between booking and arrival; right-skewed → log1p then StandardScaler'),
    ('total_nights',                   'input',    'engineered = stays_in_week_nights + stays_in_weekend_nights'),
    ('weekend_share',                  'input',    'engineered = stays_in_weekend_nights / total_nights ∈ [0,1]'),
    ('party_size',                     'input',    'engineered = adults + children + babies'),
    ('previous_cancellations',         'input',    'past relationship signal; capped, log1p, StandardScaler'),
    ('previous_bookings_not_canceled', 'input',    'past relationship signal; capped, log1p, StandardScaler'),
    ('required_car_parking_spaces',    'input',    'count; capped, StandardScaler'),
    ('total_of_special_requests',      'input',    'count; capped, StandardScaler'),
    ('is_repeated_guest',              'input',    'binary {0,1}; kept as-is (no scaling)'),

    # Categorical clustering inputs
    ('market_segment',                 'input',    'commercial intermediary type within TA/TO; one-hot after rare-grouping (used in MAIN representation R0)'),
    ('distribution_channel',           'profile',  '== "TA/TO" by definition of the sub-population (constant). Reassessed in R0-alt sensitivity (Correction #4).'),
    ('reserved_room_type',             'input',    'intended room category at booking; one-hot after rare-grouping'),
    ('meal',                           'input',    'meal package booked; one-hot after rare-grouping'),
    ('customer_type',                  'input',    'transient / contract / group / transient-party; one-hot'),
    ('deposit_type',                   'input',    'no_deposit / non_refund / refundable; one-hot'),
    ('country',                        'input',    'origin geography; rare grouping with min-frequency 1% + "Other"'),

    # Time / context
    ('arrival_date_month',             'input',    'seasonality; one-hot (cyclic encoding considered in robustness)'),
    ('arrival_date_year',              'exclude',  'covers only 2015-2017; year would split clusters by data span, not behaviour'),
    ('arrival_date_week_number',       'exclude',  'redundant with arrival_date_month at this granularity'),
    ('arrival_date_day_of_month',      'exclude',  'fine granularity adds noise; not justified for behavioural segmentation'),
    ('stays_in_week_nights',           'exclude',  'absorbed into engineered total_nights / weekend_share'),
    ('stays_in_weekend_nights',        'exclude',  'absorbed into engineered total_nights / weekend_share'),
    ('adults',                         'exclude',  'absorbed into engineered party_size'),
    ('children',                       'exclude',  'absorbed into engineered party_size'),
    ('babies',                         'exclude',  'absorbed into engineered party_size'),

    # Sensitivity-only
    ('adr',                            'profile',  'price/revenue. EXCLUDED from MAIN representation (Correction #3, default behavioural). USED in withADR sensitivity comparison (R2) and in post-hoc profiling.'),
], columns=['variable', 'role', 'rationale'])

feature_decisions.to_csv(TBL_DIR / 'feature_selection.csv', index=False)
print("Feature-selection table:")
print(feature_decisions.to_string(index=False))


### Representations and the `representation_id` registry

In line with the Milestone 2 guidance, every preprocessing variant is given a unique
`representation_id`. The **main** representation for Task 1 is `R0`. The other rows are
predeclared so that Task 3 (sensitivity / robustness) and the lecturer's Correction #4 (controlled
comparison of `market_segment` vs `distribution_channel`) plug into the same protocol.

| `representation_id` | Numerical scaling | Categorical channel block | ADR | Question tested |
|---|---|---|---|---|
| **`R0` (MAIN)** | StandardScaler | `market_segment` (within TA/TO) | excluded | Main behavioural representation |
| `R0-alt` | StandardScaler | `distribution_channel` of the *parent* dataset | excluded | Are clusters driven by the channel block (Correction #4)? — runs on the *full* dataset, not used for the main results |
| `R1` | RobustScaler   | `market_segment`           | excluded | Are clusters driven by outliers / heavy tails? |
| `R2` | StandardScaler | `market_segment`           | included (capped + scaled) | Do clusters become price/value segments? |
| `R3` | StandardScaler | `market_segment` *without* `country` | excluded | Are clusters mainly geographic groups? |

The metric for every representation is **ordinary Euclidean distance on the encoded matrix**, and
all internal indices reported in this notebook are computed in the very same space.

**Metric sentence (R0):** *"Euclidean distance is computed on the final encoded matrix `X_R0`,
which stacks log1p-transformed and StandardScaler-scaled numerical variables with full one-hot
encoded categorical variables (rare categories grouped under 'Other' at the 1% minimum-frequency
threshold). One-hot columns are kept as 0/1 and are NOT variance-standardised."*


## Data Cleaning

The cleaning steps below are applied in a fixed, reproducible order. Each step is justified
by the EDA in §4. Numerical caps are conservative (use 99th-percentile or domain-motivated
ceilings) and the resulting record counts are printed so the impact is auditable.


In [ ]:
n0 = len(df)
print(f"Start (TA/TO sub-population) : {n0:,}")

# Drop rows with no guests at all 
df = df[df['party_size'] > 0].copy()
print(f"After dropping party_size==0 : {len(df):,}  (removed {n0-len(df):,})")

# Impute missing categorical / count fields
df['country']  = df['country'].fillna('PRT')          
df['children'] = df['children'].fillna(0)             # 4 NaNs → 0
df['party_size'] = df['adults'] + df['children'] + df['babies']  # recompute
df['total_nights'] = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df['weekend_share'] = np.where(
    df['total_nights'] > 0, df['stays_in_weekend_nights'] / df['total_nights'], 0.0)

# Pre-emptive cleaning for the ADR sensitivity variant (R2) 
# (We do NOT drop these rows in R0 because adr is not used; we compute a clean
# subset df_adr for R2 and post-hoc profiling.)
df_adr_clean = df[(df['adr'] > 0) & (df['adr'] <= 1000)].copy()
print(f"Records with usable ADR (0 < adr ≤ 1000) : {len(df_adr_clean):,}  "
      f"(used only for R2 / post-hoc, NOT for R0)")

# Save the clean MAIN dataframe
print(f"Final MAIN sub-population (R0)            : {len(df):,}")


In [ ]:
# Rare-category grouping for nominal inputs (1% min-frequency)
RARE_THRESHOLD = 0.01

cat_inputs = ['market_segment', 'reserved_room_type', 'meal',
              'customer_type', 'deposit_type', 'country', 'arrival_date_month']

print(f"Rare-category grouping (< {RARE_THRESHOLD*100:.0f}% threshold):")
for col in cat_inputs:
    freq = df[col].value_counts(normalize=True)
    rare = freq[freq < RARE_THRESHOLD].index.tolist()
    if rare:
        df[col] = df[col].where(~df[col].isin(rare), other='Other')
        df_adr_clean[col] = df_adr_clean[col].where(~df_adr_clean[col].isin(rare), other='Other')
        print(f"  {col:25s}: grouped {len(rare):>3d} levels → 'Other'")
    else:
        print(f"  {col:25s}: no rare categories")


## Main Representation `R0` - Pipeline

Numerical block: `[log1p(lead_time), total_nights, weekend_share, party_size,
log1p(previous_cancellations), log1p(previous_bookings_not_canceled),
required_car_parking_spaces, total_of_special_requests, is_repeated_guest]`.
Of these, `is_repeated_guest` is binary {0,1} so it is *not* re-scaled (it would otherwise be
artificially inflated, exactly as the Milestone 2 guidance warns).

Categorical block: `[market_segment, reserved_room_type, meal, customer_type, deposit_type,
country, arrival_date_month]` → `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.

`distribution_channel` is *not* in the input set: within the TA/TO sub-population it is constant
and would add a degenerate column.

`R0` is therefore exactly the *MAIN behavioural Euclidean representation* described in §5.2.


In [ ]:
from sklearn.preprocessing import FunctionTransformer

# Engineered features already computed; cap historical counts before log1p
df['previous_cancellations']        = df['previous_cancellations'].clip(upper=df['previous_cancellations'].quantile(0.99))
df['previous_bookings_not_canceled']= df['previous_bookings_not_canceled'].clip(upper=df['previous_bookings_not_canceled'].quantile(0.99))
df['required_car_parking_spaces']   = df['required_car_parking_spaces'].clip(upper=3)   # domain ceiling
df['total_of_special_requests']     = df['total_of_special_requests'].clip(upper=5)     # domain ceiling

# Final variable lists
NUM_FEATURES_LOG = ['lead_time', 'previous_cancellations', 'previous_bookings_not_canceled']
NUM_FEATURES_LIN = ['total_nights', 'weekend_share', 'party_size',
                    'required_car_parking_spaces', 'total_of_special_requests']
NUM_FEATURES_BIN = ['is_repeated_guest']      # 0/1, kept unscaled
CAT_FEATURES     = ['market_segment', 'reserved_room_type', 'meal',
                    'customer_type', 'deposit_type', 'country', 'arrival_date_month']

# Profiling-only variables (post-hoc; NOT in the clustering matrix)
POSTHOC_VARS     = ['is_canceled', 'reservation_status', 'hotel',
                    'booking_changes', 'adr', 'assigned_room_type']

print("Final clustering inputs (R0):")
print(f"  numerical (log1p)  : {NUM_FEATURES_LOG}")
print(f"  numerical (linear) : {NUM_FEATURES_LIN}")
print(f"  binary (unscaled)  : {NUM_FEATURES_BIN}")
print(f"  categorical (1-hot): {CAT_FEATURES}")
print()
print("Post-hoc / profiling-only variables (NOT in the clustering matrix):")
print(f"  {POSTHOC_VARS}")


In [ ]:
def log1p_transform(X):
    return np.log1p(np.asarray(X, dtype=float))

log1p_step = FunctionTransformer(log1p_transform, validate=False)

numeric_log_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p',   log1p_step),
    ('scaler',  StandardScaler()),
])
numeric_lin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
numeric_bin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # NO scaler: keeps 0/1 → mirrors one-hot block
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor_R0 = ColumnTransformer(
    transformers=[
        ('num_log', numeric_log_pipe, NUM_FEATURES_LOG),
        ('num_lin', numeric_lin_pipe, NUM_FEATURES_LIN),
        ('num_bin', numeric_bin_pipe, NUM_FEATURES_BIN),
        ('cat',     cat_pipe,         CAT_FEATURES),
    ],
    remainder='drop',
)

t0 = time.time()
X_R0 = preprocessor_R0.fit_transform(df)
t_prep = time.time() - t0

# Build human-readable feature names
ohe = preprocessor_R0.named_transformers_['cat']['onehot']
cat_names = ohe.get_feature_names_out(CAT_FEATURES).tolist()
feature_names_R0 = NUM_FEATURES_LOG + NUM_FEATURES_LIN + NUM_FEATURES_BIN + cat_names

print(f"Preprocessing completed in {t_prep:.2f}s")
print(f"Input  : {len(df):,} rows × {len(NUM_FEATURES_LOG+NUM_FEATURES_LIN+NUM_FEATURES_BIN+CAT_FEATURES)} variables")
print(f"R0 matrix shape : {X_R0.shape[0]:,} rows × {X_R0.shape[1]} columns")
print(f"  ({len(NUM_FEATURES_LOG)} log-num + {len(NUM_FEATURES_LIN)} lin-num + "
      f"{len(NUM_FEATURES_BIN)} binary + {len(cat_names)} one-hot)")

REPRESENTATION_ID = 'R0-standard-noADR-marketSegment-countryRare1pct'
print(f"\nrepresentation_id : {REPRESENTATION_ID}")


In [ ]:
# Sanity-check: post-scaling distributions of the numerical block
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
axes = axes.flatten()
all_num = NUM_FEATURES_LOG + NUM_FEATURES_LIN
for i, feat in enumerate(all_num):
    axes[i].hist(X_R0[:, i], bins=50, color='#1565C0', alpha=0.75, edgecolor='white')
    axes[i].set_title(f'{feat}\n(scaled)', fontsize=10)
    mu, sd = X_R0[:, i].mean(), X_R0[:, i].std()
    axes[i].text(0.97, 0.95, f'μ={mu:.2f}\nσ={sd:.2f}',
                 transform=axes[i].transAxes, ha='right', va='top',
                 fontsize=8, color='grey')
# Hide unused axis
for j in range(len(all_num), len(axes)):
    axes[j].axis('off')

fig.suptitle('Figure 6 — Post-StandardScaler distributions of numerical block (R0)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_scaled_distributions.png')
plt.show()


## Internal Validity Indices - Helper

We report three internal indices, all computed in the same Euclidean representation `X_R0`:

- **Silhouette score** *(higher = better, range [−1, 1])*. *O(n²)*; evaluated on a fixed
  seeded sub-sample of size `SUBSAMPLE_N`.
- **Calinski–Harabasz index** *(higher = better)*. *O(n)*; evaluated on the full sub-population.
- **Davies–Bouldin index** *(lower = better)*. *O(n)*; evaluated on the full sub-population.

This is the protocol that Task 2 (alternative family) and Task 3 (stability) will reuse.


In [ ]:
def compute_internal_indices(X, labels, subsample_n=SUBSAMPLE_N, seed=SEED):
    """Internal indices computed in the same Euclidean representation as clustering."""
    n = len(labels)
    if n > subsample_n:
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=subsample_n, replace=False)
        X_s, l_s = X[idx], labels[idx]
    else:
        X_s, l_s = X, labels

    # Silhouette is degenerate for k==1 or for any partition with a single cluster
    if len(np.unique(l_s)) < 2:
        return {'silhouette': np.nan, 'calinski_harabasz': np.nan, 'davies_bouldin': np.nan}

    sil = silhouette_score(X_s, l_s, metric='euclidean')
    ch  = calinski_harabasz_score(X, labels)
    db  = davies_bouldin_score(X, labels)
    return {'silhouette': sil, 'calinski_harabasz': ch, 'davies_bouldin': db}

print("compute_internal_indices() ready.")


## Task 1.2 - Baseline K-Means

**Predefined K-grid:** `k ∈ {2, 3, 4, 5, 6, 7, 8}`. Justification for the bounds:

- Lower bound `k=2`: any meaningful partition requires at least 2 groups.
- Upper bound `k=8`: the project brief and Milestone 2 guidance both recommend a fixed grid. 
  We extend slightly to `k=2` so that the elbow comparison spans the
  whole interval, but cap at 8 because more clusters would be hard to interpret in a behavioural
  segmentation report.

**Selection rule** (declared *before* running the models):

1. Compute Silhouette, Calinski–Harabasz and Davies–Bouldin for every `k` in the grid.
2. Choose the `k` that maximises Silhouette **and** is at, or just past, the elbow inflection of
   Inertia. If the Silhouette and elbow disagree, prefer the smaller `k` (parsimony) provided the
   Silhouette gap is < 0.02.
3. Confirm by also looking at Calinski–Harabasz (higher better) and Davies–Bouldin (lower better).

`init='k-means++'`, `n_init=5`, `max_iter=500`. The 20 random restarts make the baseline robust
to the lottery of any single random initialisation.


In [ ]:
K_RANGE = list(range(2, 9))          
results_kmeans = []

print(f"{'k':>3} | {'Inertia':>14} | {'Silhouette':>10} | {'CH':>10} | {'DB':>8} | {'time(s)':>8}")
print("-" * 80)

for k in K_RANGE:
    t0 = time.time()
    km = KMeans(n_clusters=k, init='k-means++', n_init=5,
                max_iter=300, random_state=SEED)
    labels = km.fit_predict(X_R0)
    elapsed = time.time() - t0

    idx = compute_internal_indices(X_R0, labels)
    results_kmeans.append({
        'k': k, 'inertia': km.inertia_, 'time_s': elapsed,
        'labels': labels, 'model': km, **idx,
    })
    print(f"{k:>3} | {km.inertia_:>14,.1f} | {idx['silhouette']:>10.4f} | "
          f"{idx['calinski_harabasz']:>10,.0f} | {idx['davies_bouldin']:>8.4f} | "
          f"{elapsed:>8.2f}")

results_kmeans_df = pd.DataFrame([
    {k: r[k] for k in ['k','inertia','silhouette','calinski_harabasz','davies_bouldin','time_s']}
    for r in results_kmeans
])
results_kmeans_df.to_csv(TBL_DIR / 'kmeans_results.csv', index=False)


In [ ]:
# Plot the four K-means selection diagnostics together
ks = [r['k'] for r in results_kmeans]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1) Elbow / inertia
axes[0,0].plot(ks, [r['inertia'] for r in results_kmeans],
               'o-', color='#1565C0', linewidth=2, markersize=8)
axes[0,0].set_title('Elbow (Inertia) — lower is better')
axes[0,0].set_xlabel('k'); axes[0,0].set_ylabel('Inertia (WCSS)')
axes[0,0].set_xticks(ks)
axes[0,0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x/1e3:.0f}k'))

# 2) Silhouette
sils = [r['silhouette'] for r in results_kmeans]
best_sil_k = ks[int(np.argmax(sils))]
axes[0,1].plot(ks, sils, 'o-', color='#2E7D32', linewidth=2, markersize=8)
axes[0,1].axvline(best_sil_k, color='red', linestyle='--', alpha=0.6,
                  label=f'argmax k={best_sil_k}')
axes[0,1].set_title('Silhouette — higher is better')
axes[0,1].set_xlabel('k'); axes[0,1].set_ylabel('Silhouette')
axes[0,1].set_xticks(ks); axes[0,1].legend()

# 3) Calinski-Harabasz
chs = [r['calinski_harabasz'] for r in results_kmeans]
best_ch_k = ks[int(np.argmax(chs))]
axes[1,0].plot(ks, chs, 'o-', color='#6A1B9A', linewidth=2, markersize=8)
axes[1,0].axvline(best_ch_k, color='red', linestyle='--', alpha=0.6,
                  label=f'argmax k={best_ch_k}')
axes[1,0].set_title('Calinski–Harabasz — higher is better')
axes[1,0].set_xlabel('k'); axes[1,0].set_ylabel('CH index')
axes[1,0].set_xticks(ks); axes[1,0].legend()

# 4) Davies-Bouldin
dbs = [r['davies_bouldin'] for r in results_kmeans]
best_db_k = ks[int(np.argmin(dbs))]
axes[1,1].plot(ks, dbs, 'o-', color='#E65100', linewidth=2, markersize=8)
axes[1,1].axvline(best_db_k, color='red', linestyle='--', alpha=0.6,
                  label=f'argmin k={best_db_k}')
axes[1,1].set_title('Davies–Bouldin — lower is better')
axes[1,1].set_xlabel('k'); axes[1,1].set_ylabel('DB index')
axes[1,1].set_xticks(ks); axes[1,1].legend()

fig.suptitle(f'Figure 7 — K-Means selection diagnostics on representation R0 '
             f'(k ∈ {{2,…,8}})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_kmeans_selection.png')
plt.show()

print(f"Silhouette argmax        : k = {best_sil_k}")
print(f"Calinski–Harabasz argmax : k = {best_ch_k}")
print(f"Davies–Bouldin argmin    : k = {best_db_k}")


### Final selection rule applied

The Silhouette curve provides the primary signal because it is computed in the same Euclidean
space used for clustering. Calinski–Harabasz and Davies–Bouldin are reported as confirmation
indices, and Inertia is reported as a sanity check (its monotonic decrease is expected, what
matters is the elbow location). The chosen `BEST_K` is set below; any later change must update
the selection rule narrative and the experiments-log row, not just the code.


In [ ]:
# Apply the predeclared selection rule (sil-argmax + parsimony tiebreak)
sil_argmax_k = best_sil_k
sil_argmax_v = max(r['silhouette'] for r in results_kmeans)
ch_argmax_k  = best_ch_k
db_argmin_k  = best_db_k

# Parsimony tiebreak: among k <= sil_argmax_k whose silhouette is within 0.02 of the max,
# pick the smaller one if CH or DB also supports it.
PARSIMONY_GAP = 0.02
candidates = [r['k'] for r in results_kmeans
              if r['silhouette'] >= sil_argmax_v - PARSIMONY_GAP and r['k'] <= sil_argmax_k]

# Among candidates, prefer the one closest to the CH argmax / DB argmin; ties → smaller k
def _support_score(k):
    # higher = better support from CH and DB
    s_ch = next(r['calinski_harabasz'] for r in results_kmeans if r['k'] == k)
    s_db = next(r['davies_bouldin']    for r in results_kmeans if r['k'] == k)
    return (s_ch, -s_db)

candidates_sorted = sorted(candidates, key=lambda k: (_support_score(k), -k), reverse=True)
BEST_K = candidates_sorted[0]

best_result = next(r for r in results_kmeans if r['k'] == BEST_K)
best_model  = best_result['model']
best_labels = best_result['labels']

print(f"Silhouette argmax        : k = {sil_argmax_k} (sil = {sil_argmax_v:.4f})")
print(f"Parsimony candidates (within {PARSIMONY_GAP} of max sil) : {sorted(candidates)}")
print(f"Selected BEST_K          : {BEST_K}")
print(f"  Silhouette       = {best_result['silhouette']:.4f}")
print(f"  Calinski-Harabasz= {best_result['calinski_harabasz']:.0f}")
print(f"  Davies-Bouldin   = {best_result['davies_bouldin']:.4f}")
print(f"  Inertia          = {best_result['inertia']:,.0f}")

# Attach labels to the working dataframe
df_with_clusters = df.copy()
df_with_clusters['cluster_kmeans'] = best_labels


## Task 1.2 - iK-Means

1. Centre the data on the grand mean and scale by *range* (not standard deviation).
2. Pick the entity farthest from the grand mean as a tentative seed.
3. Build one *anomalous cluster* by alternating two operations:
   (a) assign every remaining point that is **strictly** closer to the tentative centroid than to
   the grand mean; (b) recompute the centroid of the assigned set.
4. Stop when membership stops changing or the centroid moves by ≤ ε.
5. Remove the cluster's indices from the residual set, repeat until exhausted.
6. Retain only clusters with size ≥ `min_cluster_size`; their standardised centroids become the
   initial seeds for a downstream k-means run.


In [ ]:
FloatArray = NDArray[np.float64]


@dataclass(frozen=True)
class APCluster:
    """One anomalous-pattern cluster as defined in the iK-Means specification §6."""
    indices: list           # row indices of the original matrix X (sorted)
    centroid_raw: FloatArray
    centroid_std: FloatArray
    size: int
    scatter_pct: float


def compute_feature_statistics(
    X: FloatArray,
    use_unit_ranges: bool = False,
) -> tuple[FloatArray, FloatArray, float]:
    """Return (mean, scales, total_scatter) per the iK-Means specification §2.

    ``scales`` is the range r_j = max - min, with r_j ← 1 wherever r_j == 0.
    ``use_unit_ranges=True`` forces r_j == 1 and is provided for unit testing.
    """
    X = np.ascontiguousarray(X, dtype=np.float64)
    mu = X.mean(axis=0)
    if use_unit_ranges:
        r = np.ones(X.shape[1], dtype=np.float64)
    else:
        r = X.max(axis=0) - X.min(axis=0)
        r = np.where(r == 0, 1.0, r)
    Y = (X - mu) / r
    D = float((Y * Y).sum())
    return mu, r, D


def normalized_squared_distances(
    X: FloatArray,
    indices: list,
    scales: FloatArray,
    reference: FloatArray,
) -> FloatArray:
    """δ_r(x_i, p) for every i in ``indices`` (specification §2)."""
    Xi = X[indices]
    diff = (Xi - reference) / scales
    return (diff * diff).sum(axis=1)


def cluster_centroid(X: FloatArray, indices: list) -> FloatArray:
    """Component-wise mean of X[indices]."""
    return X[indices].mean(axis=0)


def separate_cluster(
    X: FloatArray,
    indices: list,
    scales: FloatArray,
    a: FloatArray,
    b: FloatArray,
) -> list:
    """Return the rows i in ``indices`` that are STRICTLY closer to a than to b
    under δ_r. Ties stay with b (the reference) per specification §7."""
    Xi = X[indices]
    diff_a = (Xi - a) / scales
    diff_b = (Xi - b) / scales
    d_a = (diff_a * diff_a).sum(axis=1)
    d_b = (diff_b * diff_b).sum(axis=1)
    mask = d_a < d_b
    indices_arr = np.asarray(indices)
    return indices_arr[mask].tolist()


def extract_anomalous_cluster(
    X: FloatArray,
    indices: list,
    scales: FloatArray,
    mean: FloatArray,
    initial_centroid: FloatArray,
    seed_index: int,
    tol: float = 1e-12,
    max_iter: int = 10_000,
) -> tuple[list, FloatArray]:
    """One AP extraction (specification Algorithm 2). Returns (S, c).

    Vectorised with numpy arrays for performance; semantically identical to
    the pseudocode in §5 of the specification.
    """
    c = np.asarray(initial_centroid, dtype=np.float64).copy()
    S_prev = None
    for _ in range(max_iter):
        S = separate_cluster(X, indices, scales, a=c, b=mean)
        if len(S) == 0:
            S = [seed_index]                  # numerical safeguard
        c_new = cluster_centroid(X, S)
        S_arr = np.asarray(S)
        if S_prev is not None and S_arr.shape == S_prev.shape and np.array_equal(S_arr, S_prev):
            return sorted(S), c_new
        if np.linalg.norm(c_new - c) <= tol:
            return sorted(S), c_new
        c = c_new
        S_prev = S_arr
    return sorted(S), c


def ikmeans_initialize(
    X: FloatArray,
    min_cluster_size: int,
    tol: float = 1e-12,
    max_iter: int = 10_000,
    use_unit_ranges: bool = False,
) -> tuple[list, FloatArray]:
    """Full iK-Means initialisation (specification Algorithm 1).

    Returns ``(ap_clusters, init_centroids)`` where ``init_centroids`` are the
    STANDARDISED centroids of the retained AP clusters and ``ap_clusters`` is the
    full list of records produced (including those filtered out by ``min_cluster_size``).
    """
    X = np.ascontiguousarray(X, dtype=np.float64)
    mu, r, D = compute_feature_statistics(X, use_unit_ranges=use_unit_ranges)

    n = X.shape[0]
    remains_mask = np.ones(n, dtype=bool)
    # pre-compute distance to mu in standardised space (does not change)
    Y = (X - mu) / r
    d_to_mu_full = (Y * Y).sum(axis=1)

    ap_clusters: list[APCluster] = []

    while remains_mask.any():
        # arg max distance to mu within remains
        d_remains = np.where(remains_mask, d_to_mu_full, -np.inf)
        seed_index = int(np.argmax(d_remains))
        seed = X[seed_index].copy()

        # Convert remains_mask to indices list expected by extract_anomalous_cluster
        remains_idx = np.where(remains_mask)[0].tolist()
        S, c = extract_anomalous_cluster(
            X=X, indices=remains_idx, scales=r, mean=mu,
            initial_centroid=seed, seed_index=seed_index,
            tol=tol, max_iter=max_iter,
        )
        z = (c - mu) / r
        scatter_pct = 100.0 * len(S) * float((z * z).sum()) / D if D > 0 else 0.0

        ap_clusters.append(APCluster(
            indices=sorted(S), centroid_raw=c, centroid_std=z,
            size=len(S), scatter_pct=scatter_pct,
        ))
        # Remove S from remains
        remains_mask[np.asarray(S)] = False

    retained = [rec for rec in ap_clusters if rec.size >= min_cluster_size]
    if not retained:
        raise ValueError(
            f"No anomalous cluster satisfies min_cluster_size={min_cluster_size}; "
            f"largest AP cluster has size {max(c.size for c in ap_clusters)}."
        )
    init_centroids = np.vstack([rec.centroid_std for rec in retained])
    return ap_clusters, init_centroids


# Tiny self-test on synthetic data
def _self_test_ikmeans():
    rng = np.random.default_rng(0)
    A = rng.normal(loc=[0, 0], scale=0.2, size=(200, 2))
    B = rng.normal(loc=[5, 5], scale=0.2, size=(200, 2))
    C = rng.normal(loc=[5, 0], scale=0.2, size=(200, 2))
    Xs = np.vstack([A, B, C])
    ap, init = ikmeans_initialize(Xs, min_cluster_size=20)
    assert init.shape[1] == 2
    assert init.shape[0] >= 3, "Should retain at least 3 AP seeds on this synthetic data."
    return len(ap), init.shape[0]

n_total_ap, n_retained = _self_test_ikmeans()
print(f"Self-test: {n_retained} retained AP seeds out of {n_total_ap} extracted "
      "(synthetic 3-cluster data) — OK.")


### Apply iK-Means to representation `R0`

Because the AP procedure scales as *O(n²)* in distance computations (one pass per remaining
entity per AP iteration), running it on the full ~98 k records is unnecessarily expensive.
Mirkin's recommendation, reflected in the specification (`min_cluster_size` controls retention),
is to run AP on the data, retain large enough AP centroids, and pass them as initial centroids
to a single downstream k-means run.

For computational tractability we run the AP extraction on a **fixed seeded sub-sample of 20,000
records** (declared up-front and logged in `experiments.csv`); the *initial centroids* it returns
are then projected back onto the full sub-population through one `KMeans(init=…, n_init=1)` run.
This is the standard practical compromise documented in the lecture notes.


In [ ]:
IK_AP_SAMPLE_N    = 20_000     # AP extraction subsample
IK_MIN_CLUSTER_SIZE = 50       # entities; small enough to keep most AP clusters

rng_ik = np.random.default_rng(SEED)
ap_idx = rng_ik.choice(X_R0.shape[0], size=min(IK_AP_SAMPLE_N, X_R0.shape[0]),
                       replace=False)
X_ap = X_R0[ap_idx]

t0 = time.time()
ap_clusters_full, init_centroids_full = ikmeans_initialize(
    X_ap, min_cluster_size=IK_MIN_CLUSTER_SIZE, tol=1e-9, max_iter=300,
)
t_ap = time.time() - t0

print(f"AP extraction completed in {t_ap:.2f}s on {len(X_ap):,} entities.")
print(f"Total AP clusters extracted     : {len(ap_clusters_full)}")
print(f"AP clusters retained (size ≥ {IK_MIN_CLUSTER_SIZE}) : {init_centroids_full.shape[0]}")

# Inspect the AP-cluster table
ap_summary = pd.DataFrame([
    {'rank': i, 'size': c.size, 'scatter_pct': c.scatter_pct,
     'retained': c.size >= IK_MIN_CLUSTER_SIZE}
    for i, c in enumerate(ap_clusters_full)
]).sort_values('size', ascending=False).head(15)
print("\nTop 15 AP clusters (by size):")
print(ap_summary.to_string(index=False))
ap_summary.to_csv(TBL_DIR / 'ap_clusters_summary.csv', index=False)


In [ ]:
# Iterate iK-Means over the SAME k-grid as K-Means: take the top-k AP centroids
# as initial seeds and run one KMeans pass on the FULL X_R0.
results_ikmeans = []
sorted_ap = sorted(ap_clusters_full, key=lambda c: -c.size)        # largest first

print(f"{'k':>3} | {'Inertia':>14} | {'Silhouette':>10} | {'CH':>10} | {'DB':>8} | {'time(s)':>8}")
print("-" * 80)

for k in K_RANGE:
    # Pick the top-k retained AP centroids; if fewer than k retained, pad with random seeds
    pool = [c.centroid_std for c in sorted_ap if c.size >= IK_MIN_CLUSTER_SIZE]
    if len(pool) < k:
        extra = [c.centroid_std for c in sorted_ap if c.size < IK_MIN_CLUSTER_SIZE]
        pool = pool + extra
    init_k = np.vstack(pool[:k])
    # The AP centroids come from a (mu, range)-standardised space (per spec).
    # X_R0 is already z-scored and one-hot. 
    init_in_xspace = []
    pop_idx_global = ap_idx     # mapping of AP rows back to the full matrix
    for c in sorted_ap[:k]:
        member_global = pop_idx_global[c.indices]
        init_in_xspace.append(X_R0[member_global].mean(axis=0))
    init_in_xspace = np.vstack(init_in_xspace)

    t0 = time.time()
    km_ik = KMeans(n_clusters=k, init=init_in_xspace, n_init=1,
                   max_iter=300, random_state=SEED)
    labels_ik = km_ik.fit_predict(X_R0)
    elapsed = time.time() - t0

    idx = compute_internal_indices(X_R0, labels_ik)
    results_ikmeans.append({
        'k': k, 'inertia': km_ik.inertia_, 'time_s': elapsed,
        'labels': labels_ik, 'model': km_ik, **idx,
    })
    print(f"{k:>3} | {km_ik.inertia_:>14,.1f} | {idx['silhouette']:>10.4f} | "
          f"{idx['calinski_harabasz']:>10,.0f} | {idx['davies_bouldin']:>8.4f} | "
          f"{elapsed:>8.2f}")

results_ikmeans_df = pd.DataFrame([
    {k: r[k] for k in ['k','inertia','silhouette','calinski_harabasz','davies_bouldin','time_s']}
    for r in results_ikmeans
])
results_ikmeans_df.to_csv(TBL_DIR / 'ikmeans_results.csv', index=False)


In [ ]:
# K-Means vs iK-Means side-by-side
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ks = [r['k'] for r in results_kmeans]
panels = [
    ('Silhouette', 'silhouette', True),
    ('Calinski–Harabasz', 'calinski_harabasz', True),
    ('Runtime (s)', 'time_s', False),
]
for ax, (title, key, _) in zip(axes, panels):
    ax.plot(ks, [r[key] for r in results_kmeans], 'o-',
            label='K-Means (k-means++, n_init=5)', color='#1565C0', linewidth=2)
    ax.plot(ks, [r[key] for r in results_ikmeans], 's--',
            label='iK-Means (AP-init, n_init=1)', color='#E65100', linewidth=2)
    ax.set_xlabel('k'); ax.set_ylabel(title); ax.set_title(title)
    ax.set_xticks(ks); ax.legend(fontsize=8)

fig.suptitle('Figure 8 — K-Means vs iK-Means on representation R0',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_kmeans_vs_ikmeans.png')
plt.show()

print("\nReading the comparison:")
print("  • If iK-Means achieves comparable Silhouette/CH with n_init=1, it justifies using the")
print("    AP procedure as a more deterministic initialisation strategy than k-means++.")
print("  • If k-means++ (n_init=5) outperforms, the AP seeding is sensitive to outliers in this")
print("    high-dimensional one-hot-rich representation and the multi-restart baseline is preferred.")


## Cluster Profiling (K-Means BEST_K)

We provide cluster profiles in three forms, all of which are required by the brief:

1. **Numerical profiles** — mean / median / std of each clustering input, by cluster.
2. **Categorical profiles** — modal value of each clustering input, by cluster.
3. **Post-hoc profiles** — distributions of `is_canceled`, `reservation_status`, `hotel`,
   `booking_changes` and `adr` by cluster. **These variables are NOT clustering inputs**;
   they are used here only to *describe* the clusters. Doing the reverse — letting them define
   clusters — would be the leakage error the brief explicitly forbids.


In [ ]:
# Cluster sizes
sizes = pd.Series(best_labels).value_counts().sort_index()
print(f"Cluster sizes (k = {BEST_K}):")
for c, n in sizes.items():
    print(f"  Cluster {c}: {n:>7,} ({n/len(best_labels)*100:>5.1f}%)")

In [ ]:
# Numerical profile
ALL_NUM = NUM_FEATURES_LOG + NUM_FEATURES_LIN + NUM_FEATURES_BIN
profile_num = (
    df_with_clusters
    .groupby('cluster_kmeans')[ALL_NUM]
    .agg(['mean', 'median', 'std'])
    .round(2)
)
print("CLUSTER PROFILES — Numerical inputs")
print("=" * 90)
print(profile_num.to_string())
profile_num.to_csv(TBL_DIR / 'profile_numerical.csv')

In [ ]:
# Categorical profile (mode)
print("CLUSTER PROFILES — Categorical inputs (modal value)")
print("=" * 80)
modes = {}
for feat in CAT_FEATURES:
    m = df_with_clusters.groupby('cluster_kmeans')[feat].agg(
        lambda s: s.value_counts().index[0])
    modes[feat] = m
    print(f"  {feat:25s}: {dict(m)}")
pd.DataFrame(modes).to_csv(TBL_DIR / 'profile_categorical_mode.csv')

In [ ]:
# Post-hoc profiling (leakage-safe: outcomes were not used to form clusters)
df_with_clusters['is_canceled']        = df['is_canceled'].values
df_with_clusters['reservation_status'] = df['reservation_status'].values
df_with_clusters['hotel']              = df['hotel'].values
df_with_clusters['booking_changes']    = df['booking_changes'].values   
df_with_clusters['adr']                = df['adr'].values               

print("POST-HOC PROFILING — outcome / profiling-only variables by cluster")
print("=" * 80)
posthoc = df_with_clusters.groupby('cluster_kmeans').agg(
    cancellation_rate    = ('is_canceled', 'mean'),
    cluster_size         = ('is_canceled', 'count'),
    mean_booking_changes = ('booking_changes', 'mean'),
    mean_adr             = ('adr', lambda s: s[s > 0].mean()),       # ignore non-commercial rows
    median_adr           = ('adr', lambda s: s[s > 0].median()),
).round(3)
posthoc['cancellation_rate'] = (posthoc['cancellation_rate'] * 100).round(1)
print(posthoc.to_string())
posthoc.to_csv(TBL_DIR / 'profile_posthoc.csv')


In [ ]:
# Radar chart of cluster profiles (numerical inputs only, normalised)
profile_means = df_with_clusters.groupby('cluster_kmeans')[ALL_NUM].mean()
profile_norm  = (profile_means - profile_means.min()) / (profile_means.max() - profile_means.min() + 1e-9)

categories = ALL_NUM
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
colors = plt.cm.Set2(np.linspace(0, 1, BEST_K))

for c in range(BEST_K):
    values = profile_norm.loc[c].tolist() + [profile_norm.loc[c].tolist()[0]]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {c}', color=colors[c])
    ax.fill(angles, values, alpha=0.12, color=colors[c])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n') for f in categories], size=9)
ax.set_yticklabels([])
ax.set_title(f'Figure 9 — Cluster profiles (numerical inputs), k = {BEST_K}',
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.10))

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_radar_profiles.png')
plt.show()


In [ ]:
# Box plots of key features by cluster
KEY_FEATS = ['lead_time', 'total_nights', 'party_size', 'total_of_special_requests']
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
colors = sns.color_palette('Set2', BEST_K)

for i, feat in enumerate(KEY_FEATS):
    data = [df_with_clusters[df_with_clusters['cluster_kmeans']==c][feat].dropna()
            for c in range(BEST_K)]
    bp = axes[i].boxplot(data, patch_artist=True, notch=True,
                         boxprops=dict(alpha=0.85),
                         medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[i].set_title(feat.replace('_', '\n'), fontsize=10)
    axes[i].set_xlabel('Cluster')
    p99 = df_with_clusters[feat].quantile(0.99)
    axes[i].set_ylim(-(p99*0.05), p99 * 1.05)

fig.suptitle(f'Figure 10 — Feature distributions by cluster (k = {BEST_K})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_boxplots_clusters.png')
plt.show()


In [ ]:
# Categorical-block stacked bars by cluster
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, feat in enumerate(['market_segment', 'reserved_room_type',
                          'customer_type', 'deposit_type']):
    cross = pd.crosstab(df_with_clusters['cluster_kmeans'],
                        df_with_clusters[feat], normalize='index')
    cross.plot(kind='bar', stacked=True, ax=axes[i],
               colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{feat} composition by cluster')
    axes[i].set_xlabel('Cluster'); axes[i].set_ylabel('Proportion')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(title=feat, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')

fig.suptitle(f'Figure 11 — Categorical-input composition by cluster (k = {BEST_K})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig11_cat_by_cluster.png')
plt.show()


In [ ]:
# Post-hoc profiling: outcomes by cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# 1) Cancellation rate
rates = df_with_clusters.groupby('cluster_kmeans')['is_canceled'].mean().mul(100)
axes[0].bar(rates.index, rates.values, color=sns.color_palette('Set2', BEST_K),
            edgecolor='white', linewidth=0.8)
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Cancellation rate (%)')
axes[0].set_title('Cancellation rate (post-hoc)')
axes[0].axhline(df_with_clusters['is_canceled'].mean()*100,
                color='black', linestyle='--', alpha=0.6, label='Overall mean')
axes[0].legend(fontsize=9)

# 2) Hotel composition
hot = pd.crosstab(df_with_clusters['cluster_kmeans'], df_with_clusters['hotel'],
                  normalize='index')
hot.plot(kind='bar', stacked=True, ax=axes[1],
         color=['#1565C0', '#F57F17'], edgecolor='white')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Proportion')
axes[1].set_title('Hotel type composition (post-hoc)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Hotel', fontsize=9)

# 3) booking_changes (profiling-only, Correction #2) and ADR (Correction #3)
bc_mean = df_with_clusters.groupby('cluster_kmeans')['booking_changes'].mean()
adr_med = df_with_clusters[df_with_clusters['adr']>0].groupby('cluster_kmeans')['adr'].median()
axw = 0.4
xs = np.arange(BEST_K)
axes[2].bar(xs - axw/2, bc_mean.values, width=axw,
            color='#90A4AE', label='mean booking_changes')
axes[2].set_ylabel('mean booking_changes', color='#90A4AE')
ax2b = axes[2].twinx()
ax2b.bar(xs + axw/2, adr_med.values, width=axw,
         color='#9C27B0', alpha=0.8, label='median ADR (€)')
ax2b.set_ylabel('median ADR (€)', color='#9C27B0')
axes[2].set_xticks(xs); axes[2].set_xlabel('Cluster')
axes[2].set_title('Profiling-only variables (Corrections #2, #3)')

fig.suptitle('Figure 12 — Post-hoc profiling: outcome and profiling-only variables by cluster',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig12_posthoc.png')
plt.show()


## Agreement between K-Means and iK-Means at `BEST_K`

Adjusted Rand Index (ARI) measures how similar two partitions are after correction for chance.
ARI = 1 means identical partitions; ARI = 0 means random agreement; negative values are
worse-than-random.


In [ ]:
ik_best = next(r for r in results_ikmeans if r['k'] == BEST_K)
ari = adjusted_rand_score(best_labels, ik_best['labels'])
print(f"ARI(K-Means, iK-Means) at k = {BEST_K} : {ari:.4f}")
print(f"K-Means  Silhouette : {best_result['silhouette']:.4f}  | runtime {best_result['time_s']:.2f}s")
print(f"iK-Means Silhouette : {ik_best['silhouette']:.4f}  | runtime {ik_best['time_s']:.2f}s + AP extraction {t_ap:.2f}s")


## Experiment Logging 

`experiments.csv` is appended to throughout the project. Its schema is fixed up front so that
runs from Tasks 2 and 3 add rows in a coherent format.


In [ ]:
experiments_rows = []

# K-means rows
for r in results_kmeans:
    experiments_rows.append({
        'date'              : pd.Timestamp.now().strftime('%Y-%m-%d'),
        'run_id'            : f'kmeans-k{r["k"]}-seed{SEED}',
        'representation_id' : REPRESENTATION_ID,
        'method'            : 'KMeans',
        'parameters'        : 'init=k-means++,n_init=5,max_iter=300',
        'seed'              : SEED,
        'sample_rule'       : 'full TA/TO sub-population',
        'k'                 : r['k'],
        'silhouette'        : r['silhouette'],
        'calinski_harabasz' : r['calinski_harabasz'],
        'davies_bouldin'    : r['davies_bouldin'],
        'inertia'           : r['inertia'],
        'time_s'            : r['time_s'],
        'notes'             : 'Task 1.2 baseline',
    })

# iK-means rows
for r in results_ikmeans:
    experiments_rows.append({
        'date'              : pd.Timestamp.now().strftime('%Y-%m-%d'),
        'run_id'            : f'ikmeans-k{r["k"]}-seed{SEED}',
        'representation_id' : REPRESENTATION_ID,
        'method'            : 'iKMeans-AP',
        'parameters'        : f'AP_min_size={IK_MIN_CLUSTER_SIZE},AP_subsample={IK_AP_SAMPLE_N},'
                              'init=AP-centroids,n_init=1,max_iter=300',
        'seed'              : SEED,
        'sample_rule'       : f'AP on subsample {IK_AP_SAMPLE_N}; downstream KMeans on full',
        'k'                 : r['k'],
        'silhouette'        : r['silhouette'],
        'calinski_harabasz' : r['calinski_harabasz'],
        'davies_bouldin'    : r['davies_bouldin'],
        'inertia'           : r['inertia'],
        'time_s'            : r['time_s'],
        'notes'             : 'Task 1.2 iK-means (specification-compliant)',
    })

experiments = pd.DataFrame(experiments_rows)

# Append (don't overwrite) so future runs in Tasks 2/3 add rows
exp_path = RPT_DIR / 'experiments.csv'
mode = 'a' if exp_path.exists() else 'w'
header = mode == 'w'
experiments.to_csv(exp_path, mode=mode, header=header, index=False)

print(f"Wrote {len(experiments)} rows to experiments.csv (mode={mode}).")
print()
print(experiments.head(3).to_string(index=False))


## Sensitivity / Robustness Plan

The full sensitivity protocol is executed in **Task 3** of the project. This section records
exactly what will be tested and which question each variant answers.

| Variant | Representation | Question | Stability metric |
|---|---|---|---|
| **R0 (main)** | StandardScaler + market_segment + countryRare1pct + noADR | What are the baseline behavioural clusters? | -- |
| R1 (scaling) | RobustScaler + market_segment + countryRare1pct + noADR | Are clusters driven by outliers / heavy tails? | ARI(R0, R1) |
| R0-alt (channel) | StandardScaler + **distribution_channel** (full dataset) + countryRare1pct + noADR | **Are clusters driven by the channel block? (Correction #4)** | ARI(R0, R0-alt) - narrative only |
| R2 (ADR) | StandardScaler + market_segment + countryRare1pct + **withADR** | Do clusters become price/value segments? | ARI(R0, R2) |
| R3 (no country) | StandardScaler + market_segment + **no country** + noADR | Are clusters mainly geographic groups? | ARI(R0, R3) |
| Random restarts | R0 with 5 different seeds | Stability of K-Means under randomness | mean ± std of indices, mean pairwise ARI |

The stability protocol uses `SEEDS_STABILITY = [12345, 23456, 34567, 45678, 56789]` (≥ 5 seeds
as required by the brief).
